### Contexto e Objetivo

Está sendo desenvolvido um modelo de classificação de dados em uma base de serviço para avaliar se um cliente irá continuar no serviço ou irá cancelá-lo. Essa analise poderá ser util para identificar as chances do cliente querer desistir do serviço que possui, e avaliar os fatores que podem afetar nessa decisão


Importação de Dependências

In [51]:
import pandas as pd
import numpy as np 

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
import xgboost as xgb


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

In [11]:
df_credit = pd.read_csv(r"..\\documents\\Credit_Card_Churn.csv")

# Análise Inicial da base de Crédito

In [12]:
df_credit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CLIENTNUM                 10000 non-null  int64  
 1   Attrition_Flag            10000 non-null  object 
 2   Customer_Age              10000 non-null  int64  
 3   Gender                    10000 non-null  object 
 4   Dependent_count           10000 non-null  int64  
 5   Education_Level           10000 non-null  object 
 6   Marital_Status            10000 non-null  object 
 7   Income_Category           10000 non-null  object 
 8   Card_Category             10000 non-null  object 
 9   Months_on_book            10000 non-null  int64  
 10  Total_Relationship_Count  10000 non-null  int64  
 11  Months_Inactive_12_mon    10000 non-null  int64  
 12  Contacts_Count_12_mon     10000 non-null  int64  
 13  Credit_Limit              10000 non-null  float64
 14  Total_R

Análise de valores únicos

In [13]:
for x in df_credit.columns:
    print(f'Coluna: {x}')
    print(np.sort(df_credit[x].unique()), "\n")
    

Coluna: CLIENTNUM
[100000000 100000001 100000002 ... 100009997 100009998 100009999] 

Coluna: Attrition_Flag
['Attrited Customer' 'Existing Customer'] 

Coluna: Customer_Age
[26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49
 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72] 

Coluna: Gender
['F' 'M'] 

Coluna: Dependent_count
[0 1 2 3 4 5] 

Coluna: Education_Level
['College' 'Doctorate' 'Graduate' 'High School' 'Post-Graduate'
 'Uneducated'] 

Coluna: Marital_Status
['Divorced' 'Married' 'Single'] 

Coluna: Income_Category
['$120K +' '$40K - $60K' '$60K - $80K' '$80K - $120K' 'Less than $40K'
 'Unknown'] 

Coluna: Card_Category
['Blue' 'Gold' 'Platinum' 'Silver'] 

Coluna: Months_on_book
[12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35
 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59
 60] 

Coluna: Total_Relationship_Count
[1 2 3 4 5 6] 

Coluna: Months_Inactive_12_mon
[0 1 2 3 4 5 6] 

Coluna: 

Avaliar se nos valores numericos apresentam valores abaixo de 0

In [17]:
[c  for c in df_credit.columns if df_credit[c].dtype != 'object']

['CLIENTNUM',
 'Customer_Age',
 'Dependent_count',
 'Months_on_book',
 'Total_Relationship_Count',
 'Months_Inactive_12_mon',
 'Contacts_Count_12_mon',
 'Credit_Limit',
 'Total_Revolving_Bal',
 'Avg_Open_To_Buy',
 'Total_Amt_Chng_Q4_Q1',
 'Total_Trans_Amt',
 'Total_Trans_Ct',
 'Total_Ct_Chng_Q4_Q1',
 'Avg_Utilization_Ratio']

In [24]:
colunas_numericas = [c  for c in df_credit.columns if df_credit[c].dtype != 'object']

print("Colunas com valores abaixo de 0:\n")

for c in colunas_numericas:
    if (df_credit[c] < 0).any():
        print(c)


Colunas com valores abaixo de 0:

Avg_Open_To_Buy


In [25]:
# Análise percentual de clientes inexistentes

df_credit['Attrition_Flag'].value_counts(normalize=True)

Attrition_Flag
Existing Customer    0.85
Attrited Customer    0.15
Name: proportion, dtype: float64

Adaptação dos Dados

In [ ]:
# Nessa etapa será adaptada as colunas string que serão testadas nos modelos

df_base_credit = df_credit.copy()

income_category_map = {"Unknown": 0, "Less than $40K": 1, "$40K - $60K": 2,
                       "$60K - $80K": 3, "$80K - $120K": 4, "$120K +": 5}

education_map = {
       'Uneducated': 0, 'High School': 1, 'College': 2,
       'Graduate': 3, 'Post-Graduate': 4, 'Doctorate': 5 
}

atrition_flag_map = {
    'Existing Customer': 1,
    'Attrited Customer': 0
}

df_base_credit['Is_Male'] = df_base_credit['Gender'].map(lambda x: x == 'M').astype("int")
df_base_credit['Is_Female'] = df_base_credit['Gender'].map(lambda x: x == 'F').astype("int")

df_base_credit['Attrition_Flag'] = df_base_credit['Attrition_Flag'].map(
                                                    lambda x: atrition_flag_map[x]
                                                ).astype("int")

df_base_credit['Income_Category'] = df_base_credit['Income_Category'].map(
                                                    lambda x: income_category_map[x]
                                                ).astype("int")

df_base_credit['Education_Level'] = df_base_credit['Education_Level'].map(
                                                    lambda x: education_map[x]
                                                ).astype("int")

df_base_credit['Status_Divorced'] = df_base_credit['Marital_Status'].map(lambda x: x == 'Divorced').astype("int")
df_base_credit['Status_Single'] = df_base_credit['Marital_Status'].map(lambda x: x == 'Single').astype("int")
df_base_credit['Status_Married'] = df_base_credit['Marital_Status'].map(lambda x: x == 'Married').astype("int")

df_base_credit['Is_Blue_Card'] = df_base_credit['Card_Category'].map(lambda x: x == 'Blue').astype("int")
df_base_credit['Is_Silver_Card'] = df_base_credit['Card_Category'].map(lambda x: x == 'Silver').astype("int")
df_base_credit['Is_Gold_Card'] = df_base_credit['Card_Category'].map(lambda x: x == 'Gold').astype("int")
df_base_credit['Is_Platinum_Card'] = df_base_credit['Card_Category'].map(lambda x: x == 'Platinum').astype("int")


# A coluna CLIENTNUM será removida pois apresenta valores repetidos, 
# e então atrapalhando na classificação dos dados

df_base_credit = df_base_credit.drop(columns=['CLIENTNUM', 'Gender', 'Marital_Status', 'Card_Category'])

df_base_credit

,Attrition_Flag,Customer_Age,Dependent_count,Education_Level,Income_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Is_Male,Is_Female,Status_Divorced,Status_Single,Status_Married,Is_Blue_Card,Is_Silver_Card,Is_Gold_Card,Is_Platinum_Card
0,1,57,0,0,2,51,1,2,6,3206.56,1878.94,1327.62,1.07,13740,34,2.09,0.59,0,1,1,0,0,1,0,0,0
1,0,61,3,4,2,19,2,2,6,5134.84,2498.54,2636.30,2.65,14279,64,0.61,0.49,0,1,0,1,0,0,1,0,0
2,1,62,0,1,1,31,2,6,3,20704.64,1581.42,19123.22,2.00,19353,41,1.36,0.08,0,1,1,0,0,1,0,0,0
3,1,39,3,5,3,20,5,0,5,28157.67,552.70,27604.97,2.56,18360,123,0.71,0.02,1,0,0,0,1,1,0,0,0
4,1,52,3,3,5,24,1,5,0,27955.43,1430.73,26524.70,0.60,1205,69,0.46,0.05,0,1,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,33,1,4,2,42,2,1,3,14118.49,1608.60,12509.89,1.33,12270,76,2.71,0.11,0,1,0,1,0,1,0,0,0
9996,0,69,3,5,3,39,6,0,5,1489.53,2139.22,-649.69,1.33,9976,20,1.19,1.44,1,0,0,0,1,1,0,0,0
9997,0,46,2,0,3,49,1,3,0,18717.68,398.80,18318.88,1.09,4585,43,0.57,0.02,1,0,1,0,0,1,0,0,0
9998,1,33,3,3,5,16,5,3,1,24765.71,942.11,23823.60,2.44,17092,54,0.85,0.04,1,0,0,1,0,1,0,0,0


In [ ]:
df_base_credit[['']]

Definição das variáveis de treino e teste

In [37]:
X = df_base_credit.drop(columns=['Attrition_Flag'])
y = df_base_credit['Attrition_Flag']

# Será dividido os dados dos eixos x e y em 80% de treino e 20% de teste
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2)

# Modelo de Regressão Logística

In [53]:
# Modelo Logistic Regression
logistic_regression = LogisticRegression()
logistic_regression.fit(X_train, Y_train)

# Obtenção de Probabilidades (importante para o Gini), 
# Será pego a probabilidade de Churn (classe 1)
prob_lr = logistic_regression.predict_proba(X_test)[:, 1]

logistic_regression.score(X_train, Y_train)

c:\Users\rpf97\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8455

In [50]:
# Calculo da Curva de Lorenz e Coeviciente de Gini

# o coeficiente de gini do modelo é baixo
auc = roc_auc_score(Y_test, prob_lr)
gini = 2 * auc - 1

print(f'O Coeficiente de Gini desse modelo é {gini}')

O Coeficiente de Gini desse modelo é -0.03493396984375852


In [52]:
# Matriz de Conversão (usado para prever as saídas reais e previstes)

confusion_matrix(y, logistic_regression.predict(X))

array([[   0, 1538],
       [   0, 8462]])

In [55]:
# Análise de quais colunas estão afetando o Modelo

pd.DataFrame({
    'Coluna': X_train.columns,
    'Coeficiente': logistic_regression.coef_[0]   
}).sort_values(by='Coeficiente', ascending=True)


,Coluna,Coeficiente
3,Income_Category,-4.29e-03
10,Avg_Open_To_Buy,-1.20e-05
8,Credit_Limit,1.35e-05
12,Total_Trans_Amt,2.19e-05
9,Total_Revolving_Bal,2.55e-05
22,Is_Silver_Card,6.07e-04
18,Status_Divorced,9.90e-04
13,Total_Trans_Ct,1.59e-03
16,Is_Male,1.60e-03
23,Is_Gold_Card,1.62e-03


# Random Florest

In [16]:
# Modelo Random Florest
random_florest = RandomForestClassifier()
random_florest.fit(X_train, Y_train)

prob_rf = random_florest.predict_proba(X_test)[:, 1]
prob_rf

array([0.78, 0.81, 0.84, ..., 0.84, 0.88, 0.74], shape=(3000,))

# XGBOOT

In [ ]:
#xgboot_classifier = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1) 

#xgboot_classifier.fit(X_train, Y_train)

#prob_xgb = xgboot_classifier.predict_proba(X_test)[:, 1]
#prob_xgb

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['Attrited Customer' 'Existing Customer']